# Session 3: SOLID Principles & AI Validation

> Apply the 5 core design principles to prevent technical debt and build maintainable systems.

## 1. Single Responsibility & Open/Closed

### Single Responsibility Principle (SRP)
**"A class should have only one reason to change."**

A *reason to change* = a stakeholder whose requirement could force a rewrite. If marketing changes the email template **and** the DB team changes the schema, a class that handles both will be broken twice. SRP says each class should be breakable by exactly one team/requirement.

**The class-explosion trade-off**
Applying SRP strictly does create more classes. This is intentional and manageable:
- More classes → each is shorter, simpler, and independently testable.
- Use an *orchestrator* (a thin coordinator class, or a function) to wire them together in one place. The orchestrator has no logic itself — it just calls the small pieces in order.
- Compare: one 200-line class that's hard to test vs. four 30-line classes that can each be unit-tested in isolation.

**Rule of thumb:** if you struggle to name a class with a single noun phrase (e.g., `UserEmailAndDatabaseManager`), it's doing too much.

### Open/Closed Principle (OCP)
**"Open for extension, closed for modification."**

Adding a new discount type should not require editing existing `if/elif` chains. Instead, create a new subclass of `PricingStrategy`. The calling code (`calculate(strategy, price)`) never changes. This is especially valuable in code that other teams depend on — they can extend without breaking your stable API.

In [ ]:
# ❌ SRP Violation — UserManager does too much
class UserManager:
    def create_user(self, data): pass
    def send_welcome_email(self, user): pass  # Not its job
    def save_to_database(self, user): pass    # Not its job

# ✅ SRP Compliant
class UserService:
    def create(self, data): return {'id': 1, **data}

class EmailService:
    def send_welcome(self, user): print(f'Email sent to {user["email"]}')

class UserRepository:
    def save(self, user): print(f'Saved user {user["id"]}')

# ✅ OCP — Open for extension, closed for modification
from abc import ABC, abstractmethod
class PricingStrategy(ABC):
    @abstractmethod
    def calculate(self, price: float) -> float: pass

class StandardPricing(PricingStrategy):
    def calculate(self, price): return price

class EnterpriseDiscount(PricingStrategy):
    def calculate(self, price): return price * 0.75

strategy = EnterpriseDiscount()
print(f'Enterprise price: ${strategy.calculate(1000)}')

## 💡 Additional Examples: SRP & OCP

The examples below show three common real-world scenarios where SOLID violations cause pain, and how to fix them.

**Example 1 — SRP: Report pipeline**
A single `ReportService` that generates, formats, and emails a report has three reasons to change: product changes the data model, design changes the PDF layout, ops changes the email provider. Splitting it into `ReportGenerator`, `PdfFormatter`, and `ReportMailer` means each change touches exactly one file. An orchestrator function sequences the three steps.

**Example 2 — OCP: Shape area calculator**
Adding a `Hexagon` shape should not require editing `calculate_total_area`. By making `Shape` an abstract base class, you can add unlimited shapes without touching existing code — the function loop `sum(s.area() for s in shapes)` works forever.

**Example 3 — LSP + DIP: Notification system**
`NotificationService` should work identically whether you inject `EmailSender`, `SmsSender`, or `SlackSender`. This is LSP: any subclass can substitute the base without surprising the caller. DIP is satisfied because `NotificationService.__init__` accepts the *abstraction* (`NotificationSender`), not a concrete type — so in tests you can pass a `FakeSender` that records calls instead of hitting real APIs.

In [ ]:
# ─── Example 1: SRP — Split Report processing into separate classes ───────────
# ❌ SRP Violation: ReportService handles generation, formatting, AND emailing
class ReportServiceBad:
    def generate(self, data): return {'rows': data, 'total': len(data)}
    def format_as_pdf(self, report): return f'[PDF bytes for {len(report["rows"])} rows]'
    def send_by_email(self, pdf, recipient): print(f'Emailing {len(pdf)} chars to {recipient}')

# ✅ SRP Compliant: each class has exactly one responsibility
class ReportGenerator:
    def generate(self, data: list) -> dict:
        return {'rows': data, 'total': len(data), 'avg': sum(data) / len(data) if data else 0}

class PdfFormatter:
    def format(self, report: dict) -> bytes:
        content = f"Total: {report['total']}, Avg: {report['avg']:.2f}"
        return content.encode()

class ReportMailer:
    def send(self, pdf_bytes: bytes, recipient: str):
        print(f'✉️  Sent {len(pdf_bytes)}-byte PDF to {recipient}')

# Orchestration layer wires the pieces together
data = [120, 340, 80, 220]
report = ReportGenerator().generate(data)
pdf = PdfFormatter().format(report)
ReportMailer().send(pdf, 'manager@company.com')

# ─── Example 2: OCP — Shape Area Calculator ──────────────────────────────────
from abc import ABC, abstractmethod
import math

class Shape(ABC):
    @abstractmethod
    def area(self) -> float: ...

class Circle(Shape):
    def __init__(self, radius: float): self.radius = radius
    def area(self) -> float: return math.pi * self.radius ** 2

class Rectangle(Shape):
    def __init__(self, w: float, h: float): self.w, self.h = w, h
    def area(self) -> float: return self.w * self.h

class Triangle(Shape):
    def __init__(self, base: float, height: float): self.base, self.height = base, height
    def area(self) -> float: return 0.5 * self.base * self.height

# Adding a new shape never requires modifying calculate_total_area
def calculate_total_area(shapes: list[Shape]) -> float:
    return sum(s.area() for s in shapes)

shapes = [Circle(5), Rectangle(4, 6), Triangle(3, 8)]
for s in shapes:
    print(f'{s.__class__.__name__}: area = {s.area():.2f}')
print(f'Total area: {calculate_total_area(shapes):.2f}')

# ─── Example 3: LSP + DIP — Notification System ──────────────────────────────
class NotificationSender(ABC):
    @abstractmethod
    def send(self, recipient: str, message: str) -> bool: ...

class EmailSender(NotificationSender):
    def send(self, recipient, message):
        print(f'📧 Email → {recipient}: {message}')
        return True

class SmsSender(NotificationSender):
    def send(self, recipient, message):
        print(f'📱 SMS → {recipient}: {message[:160]}')  # SMS character limit
        return True

class SlackSender(NotificationSender):
    def send(self, recipient, message):
        print(f'💬 Slack → #{recipient}: {message}')
        return True

# DIP: NotificationService depends on abstraction, not on concrete implementations
class NotificationService:
    def __init__(self, sender: NotificationSender):
        self._sender = sender

    def notify(self, recipient: str, message: str) -> bool:
        return self._sender.send(recipient, message)

# LSP: any subclass can substitute NotificationSender without breaking behavior
for sender in [EmailSender(), SmsSender(), SlackSender()]:
    svc = NotificationService(sender)
    svc.notify('devteam', 'Deployment completed successfully 🚀')


## 2. LSP, ISP, DIP

### Liskov Substitution Principle (LSP)
**"Subclasses must be substitutable for their base class without changing the correctness of the program."**

The classic failure: `Square extends Rectangle`. Callers that do `rect.set_width(5); rect.set_height(10); assert rect.area() == 50` will break when a `Square` is passed, because a square must keep width == height. The fix: model them separately, or use a shared `Shape` base that has only what both truly share.

The practical test: if you need an `if isinstance(x, SubClass)` check inside code that is supposed to work on the base class, LSP is violated.

### Interface Segregation Principle (ISP)
**"No class should be forced to implement methods it does not use."**

In Python this surfaces when a base class defines 6 abstract methods but a concrete class only needs 2 — it has to implement the other 4 as `raise NotImplementedError`. The fix: split the interface into focused, small protocols. IoT devices are a classic example: a thermostat should never have to stub out a `take_photo` method.

### Dependency Inversion Principle (DIP)
**"High-level modules must not depend on low-level modules. Both should depend on abstractions."**

Practical impact: if `UserService` imports `PostgresUserRepository` directly, you cannot unit-test `UserService` without a running database. By depending on `UserRepositoryPort` (an ABC), you can inject `InMemoryUserRepository` in tests and `PostgresUserRepository` in production — the service code is identical in both cases. This is the foundation of testable architecture.

**Liskov Substitution (LSP):** Subclasses must be substitutable for their parent without breaking behavior.

**Interface Segregation (ISP):** No class should be forced to implement interfaces it doesn't use. Split large interfaces.

**Dependency Inversion (DIP):** Depend on abstractions, not concretions. High-level modules must not depend on low-level modules.

In [ ]:
# ─── Example: LSP — Bird Hierarchy ───────────────────────────────────────────
from abc import ABC, abstractmethod

class Bird(ABC):
    @abstractmethod
    def move(self) -> str: ...

class FlyingBird(Bird):
    def move(self) -> str: return 'flying'

class SwimmingBird(Bird):
    def move(self) -> str: return 'swimming'

class Eagle(FlyingBird):
    def move(self): return f'Eagle is {super().move()} at high altitude'

class Penguin(SwimmingBird):
    def move(self): return f'Penguin is {super().move()} in cold water'

# LSP: every subclass substitutes Bird without breaking caller behavior
for bird in [Eagle(), Penguin()]:
    print(bird.move())  # ✅ No NotImplementedError like the classic Square(Rectangle) trap

# ─── Example: ISP — IoT Device Interfaces ────────────────────────────────────
# ❌ ISP Violation: one bloated interface forces all devices to implement everything
class SmartDeviceBloated(ABC):
    @abstractmethod
    def get_temperature(self): ...
    @abstractmethod
    def take_photo(self): ...   # A thermostat has no camera!
    @abstractmethod
    def unlock_door(self): ...  # A camera does not unlock doors!

# ✅ ISP Compliant: split into focused capability interfaces
class TemperatureSensor(ABC):
    @abstractmethod
    def get_temperature(self) -> float: ...

class Camera(ABC):
    @abstractmethod
    def take_photo(self) -> bytes: ...

class SmartLock(ABC):
    @abstractmethod
    def unlock(self, pin: str) -> bool: ...

class ThermostatDevice(TemperatureSensor):
    def get_temperature(self) -> float: return 22.5

class SecurityCamera(Camera, TemperatureSensor):
    def take_photo(self) -> bytes: return b'<jpeg data>'
    def get_temperature(self) -> float: return 28.0

thermostat = ThermostatDevice()
camera = SecurityCamera()
print(f'Thermostat temp: {thermostat.get_temperature()}°C')
print(f'Camera temp: {camera.get_temperature()}°C, photo: {len(camera.take_photo())} bytes')

# ─── Example: DIP — Repository Pattern ───────────────────────────────────────
class UserRepositoryPort(ABC):
    @abstractmethod
    def find_by_id(self, user_id: int) -> dict | None: ...
    @abstractmethod
    def save(self, user: dict) -> dict: ...

class InMemoryUserRepository(UserRepositoryPort):
    def __init__(self): self._store = {}
    def find_by_id(self, user_id): return self._store.get(user_id)
    def save(self, user):
        self._store[user['id']] = user
        return user

# Simulated production repository (PostgreSQL)
class PostgresUserRepository(UserRepositoryPort):
    def find_by_id(self, user_id): return {'id': user_id, 'name': 'DB User', 'source': 'postgres'}
    def save(self, user): print(f'INSERT INTO users ... {user["name"]}'); return user

# UserService does not know (or care) which backend is used
class UserService:
    def __init__(self, repo: UserRepositoryPort):  # ← Depend on abstraction
        self._repo = repo

    def get_or_create(self, user_id: int, name: str) -> dict:
        user = self._repo.find_by_id(user_id)
        if not user:
            user = self._repo.save({'id': user_id, 'name': name})
        return user

# Use InMemory for tests, Postgres in production — service code stays identical
svc = UserService(InMemoryUserRepository())
u = svc.get_or_create(1, 'Alice')
print(f'Created: {u}')
u_again = svc.get_or_create(1, 'Alice')
print(f'Fetched again: {u_again}')


## 3. AI Lab: Codebase SOLID Audit

### 🧪 AI Lab / Practice

> **TODO:** Pick a class from your current project (50+ lines) → Paste into Claude with: 'Analyze for SOLID violations. List each violation with: principle violated, line number, why it violates, and a refactored version.' → Fix the top 2 violations and submit the diff.

In [ ]:
# TODO: Implement your solution here
raise NotImplementedError('Not implemented yet — complete this exercise')